# Mini-Projet : Oreiller connecté - Comment évaluer votre sommeil ?

Compte tenu du mode de vie actuel, les gens dorment sans tenir compte des bienfaits du sommeil pour le corps humain. Pour aider à comprendre la relation entre le stress et le sommeil et pour concrétiser pleinement l'idée du « sommeil intelligent », on considère un oreiller connecté.  

L'oreiller capture des informations diverses :
- niveau de ronflement de l'utilisateur,
-  fréquence respiratoire,
-  température corporelle,
-  fréquence des mouvements des membres,
-  taux d'oxygène dans le sang,
-  mouvements oculaires,
-  nombre d'heures de sommeil,
-  fréquence cardiaque


L'objectif est de créer un modèle de classification analysant les changements physiologiques qui se produisent pendant le sommeil ainsi que les habitudes de sommeil est proposé. Sur la base de ces changements pendant le sommeil, une prédiction du stress pour le lendemain est proposée : 0 - faible/normal, 1 - moyen-faible, 2 - moyen, 3 - moyen-élevé, 4 - élevé.



In [2]:
# Librairies utiles
import numpy as np
import sklearn
import pandas as pd
import matplotlib.pyplot as plt

# Chargement des données

In [24]:
# Chargement des données d'apprentissage
DataBrut = pd.read_csv("SaYoPillow.csv")
# print(DataBrut.describe()) #On trouve le min max de chaque variable
# print(DataBrut["sl"].value_counts()) #On trouve la distribution de chaque classe
DataBrut=np.array(DataBrut)
# print('Dimension de la matrice des données :',DataBrut.shape)
Label=DataBrut[:,8]
print("label :",Label)
#On met 8 car on veut utiliser les 8 variables 
Data=DataBrut[:,:8]
print(Data)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
label=le.fit_transform(Label)

label : [3. 3. 1. 3. 0. 1. 0. 0. 0. 1. 2. 1. 0. 4. 3. 1. 1. 3. 3. 2. 2. 2. 2. 3.
 3. 1. 2. 1. 3. 2. 0. 4. 0. 4. 2. 3. 0. 4. 0. 4. 2. 2. 2. 0. 2. 4. 2. 4.
 1. 4. 0. 1. 3. 0. 1. 2. 4. 3. 3. 1. 4. 3. 0. 2. 4. 1. 0. 2. 4. 0. 1. 4.
 1. 0. 0. 4. 4. 1. 3. 4. 1. 1. 1. 4. 2. 1. 1. 1. 3. 4. 2. 3. 2. 4. 0. 2.
 4. 2. 2. 3. 4. 4. 0. 4. 1. 2. 2. 2. 4. 1. 2. 4. 0. 3. 2. 4. 2. 2. 0. 1.
 1. 3. 4. 1. 3. 4. 4. 2. 1. 4. 2. 0. 4. 3. 1. 0. 0. 2. 4. 0. 2. 3. 2. 0.
 2. 0. 0. 0. 0. 1. 3. 0. 4. 4. 0. 2. 3. 2. 4. 4. 4. 2. 1. 2. 4. 2. 0. 0.
 4. 4. 1. 2. 2. 1. 1. 1. 1. 4. 3. 1. 1. 1. 2. 1. 0. 2. 0. 2. 1. 2. 1. 1.
 4. 3. 4. 0. 0. 0. 2. 3. 0. 0. 0. 2. 4. 0. 2. 4. 1. 1. 1. 3. 0. 0. 3. 3.
 0. 2. 2. 2. 1. 4. 0. 1. 3. 4. 2. 1. 4. 0. 3. 4. 1. 3. 0. 0. 3. 4. 3. 1.
 4. 3. 3. 4. 3. 0. 3. 2. 3. 3. 1. 4. 2. 3. 1. 3. 2. 3. 4. 2. 4. 3. 2. 0.
 0. 1. 2. 3. 2. 2. 4. 4. 1. 0. 4. 1. 2. 3. 4. 3. 3. 3. 0. 3. 0. 1. 3. 3.
 3. 2. 4. 0. 4. 0. 4. 0. 2. 1. 2. 1. 3. 0. 0. 3. 2. 2. 4. 2. 3. 0. 3. 2.
 1. 0. 1. 0. 3. 3. 3. 0. 3. 4. 2. 4. 3. 4. 

In [26]:
# Séparation de l'ensemble de données en Base d'apprentissage et base de test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(Data, label, random_state=0,test_size=0.3)

print('Dimension Base d apprentissage :', X_train.shape)
print('Dimension Base de test :', X_test.shape)

Dimension Base d apprentissage : (441, 8)
Dimension Base de test : (189, 8)


# Fonctions pour réaliser une table de correspondance entre le label des approches non supervisées et le label réel

In [3]:
def retrieve_info(cluster_labels,label):
 #Associe l'étiquette la plus probable à chaque groupe dans le modèle KMeans.
 #Résultats : dictionnaire des clusters associés à chaque étiquette.

# Initialisation
  reference_labels = np.zeros((len(np.unique(cluster_labels)),1))
# Loop pour chaque label 
  for i in range(len(np.unique(cluster_labels))):
    index = np.where(cluster_labels == i,1,0)
    num = np.bincount(label[index==1]).argmax()
    reference_labels[i] = num
  return reference_labels

def correspondance(y_pred,label):
  # Correspondance entre la partition issue de la méthode non supervisée y_pred et les classes de la vérité terrain y_train
  reference_labels = retrieve_info(y_pred,label)
  label=np.array(label)
  number_labels = np.zeros(len(y_pred))
  for i in range(len(y_pred)):
    number_labels[i] = reference_labels[y_pred[i]]
  return number_labels

## Partie I : Classification du niveau de stress dans la journée sans prétraitement 

Réaliser une classification du niveau de stress par une méthode d'apprentissage supervisé et une méthode d'apprentissage non supervisé en utilisant les codes des TP précédents.

Pour chaque résultat obtenu, calculer la matrice de confusion et analyser les résultats. 

## Méthodes supervisées  : 
indiquez 3 méthodes supervisées et les paramètres utilisés.  Calculez la matrice de confusion associée et commentez les résultats

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
modele1 = QuadraticDiscriminantAnalysis(reg_param=0.1)

In [29]:
from sklearn.metrics import confusion_matrix, accuracy_score

modele1.fit(X_train, y_train)
y_pred = modele1.predict(X_test)
print("Modele 1 :", accuracy_score(y_test, y_pred))
print("Modele 1 : ",confusion_matrix(y_test, y_pred))

LinAlgError: The covariance matrix of class 0 is not full rank. Increase the value of `reg_param` to reduce the collinearity.

## Méthodes non supervisées :

**Rappels :** 
- En apprentissage non supervisé, on utilise l'ensemble des données. 
- Comme les labels ne sont pas utilisés, il faut faire une correspondance entre le numéro de la classe définie par la méthode non supervisée et le niveau réel de stress associé. Pour cela, vous utiliserez la fonction *correspondance*.

Indiquez les 2 méthodes utilisées et ses paramètres. Calculer la matrice de confusion et analyser les résutlats. 

# Partie 2 : Classification du niveau de stress avec prétraitement par ACP

- Réalisez maintenant une réduction de dimension par ACP. Expliquez quelle est la dimension la + intéressante.
- Appliquez les mêmes méthodes supervisées et non supervisées réalisées en TP et commentez les paramètres choisis. 
- Commentez les résultats. 

## Méthodes supervisées :

## Méthodes non supervisées :

# Commentaires à retenir pour le rapport 